MD# Lab 9.3: Causal Recourse and Recourse That Lasts

Chapter 9. Part A uses an invented structural causal model, so that we know the true causes, and compares three kinds of advice: a counterfactual that treats features as independent, a causal intervention, and a cheap proxy change. Part B returns to German Credit and asks how many recourses survive when the model is retrained. Run the cells in order.

MD## Part A. A world where we know the causes

**Checkpoint 1:** the fitted weights are about 1.28 (income), 0.43 (savings) and 0.19 (telephone), intercept -6.22: positive weight for the telephone, which causes nothing in this world. The repayment rate is 0.541.

In [1]:
import numpy as np
rng = np.random.default_rng(0)
n = 5000
# Structural causal model (invented, for teaching):
#   income  := U_I                         (thousands per month)
#   savings := 2 * income + U_S            (thousands)
#   phone   := 1 if income + U_P > 3       (a registered telephone: caused by income, causes nothing)
#   repay   := 1 if 0.9 * income + 0.2 * savings + U_R > 3.7
income = rng.gamma(4, 0.8, n)
savings = 2 * income + rng.normal(0, 1, n)
phone = (income + rng.normal(0, 1, n) > 3).astype(float)
repay = (0.9 * income + 0.2 * savings + rng.logistic(0, 0.6, n) > 3.7).astype(int)
F = np.c_[income, savings, phone]
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression().fit(F, repay)
w, b = clf.coef_[0], clf.intercept_[0]
print("model weights (income, savings, phone):", w.round(2), "intercept", round(b, 2))
print("repayment rate:", repay.mean().round(3))

model weights (income, savings, phone): [1.28 0.43 0.19] intercept -6.22
repayment rate: 0.541


MD### Three pieces of advice for one declined applicant

Applicant: income 2.5k, savings 5k, no telephone. For each piece of advice we report the model's new score and the applicant's *true* probability of repaying in the structural model.

**Checkpoint 2:** model score 0.30, true repayment probability 0.32. Income must rise by 0.66k if savings are held fixed but only 0.39k if savings follow (model score 0.50, true probability 0.53 after the intervention). Registering a telephone moves the model score to 0.34 and leaves the true probability at 0.32.

In [2]:
from scipy.special import expit
x = np.array([2.5, 5.0, 0.0])
score = lambda f: expit(f @ w + b)
true_p = lambda inc, sav: expit((0.9 * inc + 0.2 * sav - 3.7) / 0.6)
gap = -(x @ w + b)
print(f"model score now {score(x):.2f}, true repayment probability {true_p(2.5, 5.0):.2f}")
d_fixed = gap / w[0]                                  # raise income, hold savings fixed
d_causal = gap / (w[0] + 2 * w[1])                    # do(income), savings follow
print(f"income rise needed, savings held fixed: {d_fixed:.2f}k; with savings following: {d_causal:.2f}k")
inc = 2.5 + d_causal
print(f"causal recourse -> model score {score(np.array([inc, 2 * inc, 0.0])):.2f}, true {true_p(inc, 2 * inc):.2f}")
xp = x.copy(); xp[2] = 1.0
print(f"register a phone -> model score {score(xp):.2f}, true {true_p(2.5, 5.0):.2f}")

model score now 0.30, true repayment probability 0.32
income rise needed, savings held fixed: 0.66k; with savings following: 0.39k
causal recourse -> model score 0.50, true 0.53
register a phone -> model score 0.34, true 0.32


MD### Gaming or improvement across many applicants

**Checkpoint 3:** for the 2,361 declined applicants, giving everyone a telephone raises the mean model score from 0.172 to 0.191 and leaves the mean true repayment probability at 0.179.

In [3]:
dec = score(F) < 0.5
Fp = F[dec].copy(); Fp[:, 2] = 1.0
print("declined:", dec.sum())
print("mean model score before / after phone:", score(F[dec]).mean().round(3), score(Fp).mean().round(3))
print("mean true repayment before / after phone:", true_p(F[dec, 0], F[dec, 1]).mean().round(3), "(unchanged by construction)")

declined: 2361
mean model score before / after phone: 0.172 0.191
mean true repayment before / after phone: 0.179 (unchanged by construction)


MD## Part B. Recourse after the model is retrained

We rebuild the German Credit model and search from Lab 9.2, then retrain the model on 20 bootstrap resamples of the training data and check how many of the recourses still lead to approval. A safety margin asks the search for a score above the threshold. This part takes about three minutes.

**Checkpoint 4:** without a margin about 64 percent of recourses survive retraining; with a margin of 0.10 about 77 percent, at a higher median cost.

In [4]:
import itertools, warnings
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

def load_german():
    """German Credit (Statlog) from OpenML; falls back to data/german_credit.csv if offline."""
    try:
        from sklearn.datasets import fetch_openml
        df = fetch_openml("credit-g", version=1, as_frame=True).frame.copy()
    except Exception:
        df = pd.read_csv("../../../data/german_credit.csv")
    y = (df.pop("class").astype(str) == "good").astype(int)
    for c in df.columns:
        if str(df[c].dtype) in ("category", "object"):
            df[c] = df[c].astype(str)
    return df, y

X, y = load_german()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
CATS = [c for c in X.columns if X[c].dtype == object]

def make_model(seed=0):
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATS)], remainder="passthrough")
    return make_pipeline(pre, HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                                             max_iter=200, random_state=seed))

THR = 0.6                      # approve when the predicted probability of "good" is at least 0.6
model = make_model().fit(Xtr, ytr)
p = model.predict_proba(Xte)[:, 1]
declined = Xte[p < THR]
print(f"test AUC {roc_auc_score(yte, p):.3f}; declined {len(declined)} of {len(Xte)}")

test AUC 0.847; declined 87 of 300


In [5]:
ORDERED = {"checking_status": ["<0", "0<=X<200", ">=200"],
           "savings_status": ["<100", "100<=X<500", "500<=X<1000", ">=1000"],
           "employment": ["unemployed", "<1", "1<=X<4", "4<=X<7", ">=7"]}
IMMUTABLE = ["age", "personal_status", "foreign_worker", "num_dependents", "credit_history", "purpose",
             "job", "housing", "property_magnitude", "residence_since", "employment"]
RANGE = {c: float(Xtr[c].max() - Xtr[c].min()) for c in X.columns if X[c].dtype != object}

def cost(row, changes):
    """Range-normalized distance: numeric |change| / range; ordered category steps / (levels - 1); other 1."""
    total = 0.0
    for f, v in changes.items():
        if f in RANGE:
            total += abs(v - row[f]) / RANGE[f]
        elif f in ORDERED and row[f] in ORDERED[f] and v in ORDERED[f]:
            lv = ORDERED[f]; total += abs(lv.index(v) - lv.index(row[f])) / (len(lv) - 1)
        else:
            total += 1.0
    return total

def moves(row, free):
    """Candidate one-feature changes. free=True: anything; free=False: the actionable set."""
    out = []
    for f in X.columns:
        old = row[f]
        if free:
            if X[f].dtype == object:
                out += [(f, v) for v in sorted(Xtr[f].unique()) if v != old]
            elif f == "credit_amount":
                out += [(f, round(old * k)) for k in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.2, 1.5)]
            else:
                st = {"duration": 6, "age": 5}.get(f, 1)
                out += [(f, old + k * st) for k in range(-4, 5) if k and Xtr[f].min() <= old + k * st <= Xtr[f].max()]
            continue
        if f in IMMUTABLE:
            continue
        if f == "credit_amount":
            out += [(f, round(old * k)) for k in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9)]
        elif f == "duration":
            out += [(f, old + d) for d in (-24, -18, -12, -6, 6, 12) if 4 <= old + d <= 72]
        elif f in ("installment_commitment", "existing_credits"):
            out += [(f, old - d) for d in (1, 2, 3) if old - d >= 1]
        elif f in ("checking_status", "savings_status"):
            lv = ORDERED[f]
            if old in lv:
                out += [(f, v) for v in lv[lv.index(old) + 1:]]
            if f == "checking_status" and old != "no checking":
                out.append((f, "no checking"))        # closing the account: possible, but not an improvement
        elif f == "other_payment_plans" and old != "none":
            out.append((f, "none"))
        elif f == "other_parties" and old == "none":
            out += [(f, "guarantor"), (f, "co applicant")]
        elif f == "own_telephone" and old == "none":
            out.append((f, "yes"))                     # cheap and possible, but not an improvement
    return out

def search(mdl, row, free, thr=THR, n=1):
    """Exhaustive over one and two changes, and three changes among the 12 strongest single moves."""
    c = moves(row, free)
    def batch(combos):
        R = pd.DataFrame([row.to_dict()] * len(combos))
        for i, combo in enumerate(combos):
            for f, v in combo:
                R.at[i, f] = v
        return mdl.predict_proba(R[X.columns])[:, 1]
    one = [(mv,) for mv in c]
    p1 = batch(one)
    top = np.argsort(-p1)[:12]
    two = [(c[i], c[j]) for i, j in itertools.combinations(range(len(c)), 2) if c[i][0] != c[j][0]]
    three = [tuple(c[i] for i in t) for t in itertools.combinations(top, 3) if len({c[i][0] for i in t}) == 3]
    allc = one + two + three
    pp = np.concatenate([p1, batch(two), batch(three)])
    valid = sorted(((cost(row, dict(cb)), cb, q) for cb, q in zip(allc, pp) if q >= thr), key=lambda t: t[0])
    return valid[:n] if n else valid

In [6]:
rng = np.random.default_rng(0)
models = []
for b_ in range(20):
    idx = rng.integers(0, len(Xtr), len(Xtr))
    models.append(make_model(seed=b_).fit(Xtr.iloc[idx], ytr.iloc[idx]))
for margin in (0.0, 0.05, 0.10, 0.15):
    res = [search(model, r, False, thr=THR + margin)[0] for _, r in declined.iterrows()]
    R = []
    for (_, r), o in zip(declined.iterrows(), res):
        rr = r.copy()
        for f, v in o[1]:
            rr[f] = v
        R.append(rr)
    R = pd.DataFrame(R)[X.columns]
    valid = np.mean([(mm.predict_proba(R)[:, 1] >= THR).mean() for mm in models])
    print(f"margin {margin:.2f}: still approved {valid:.3f}, median cost {np.median([o[0] for o in res]):.3f}")

margin 0.00: still approved 0.638, median cost 0.203


margin 0.05: still approved 0.699, median cost 0.265


margin 0.10: still approved 0.772, median cost 0.353


margin 0.15: still approved 0.826, median cost 0.676


MD## Your turn

Pick one applicant whose recourse fails under several retrained models. Print the scores the 20 models give to the recourse. Is the applicant close to the threshold under all of them, or do the models disagree sharply? Write three sentences on what a lender should promise an applicant who follows the recourse.